In [ ]:
from utils_edited import *
from run_edited import *
from analysis_visualize import *

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')  # don't change, gpu 0 must have enough space for spending outside raytune
os.environ["TUNE_WARN_EXCESSIVE_EXPERIMENT_CHECKPOINT_SYNC_THRESHOLD_S"] = "0"  # don't change

# Ray Tune on selected GPUs index
# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,....."
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
gpu_per_task = 1  # 0.5 means each trial uses half a GPU

# CDR task
source_domain = "movie"
target_domain = "music"
dataset = "amazon"

# Hyperparameter sensitivity of the main model
lrlist = [0.0005, 0.001, 0.005]
clusterlist = [64, 128, 256]
# rho = 0 / 1   # association-matrix weight, you can modify it to list for grid search, current use 0 / 1 for visuzlization purpose
rho = 1.0
# {ot_weightlist, align_weightlist: 0.005, 0.01} for best movie-music visualization results.
ot_weightlist = [0.001, 0.005, 0.01]
align_weightlist = [0.005, 0.025, 0.05]
skip_ot = False  # If True: replace balanced OT with row-wise softmax correspondence
prompt_mode = "softmax" if skip_ot else "ot"
rho_tag = str(rho).replace("-", "m").replace(".", "p")

best_result = main(
    source_domain, target_domain, lrlist, rho, clusterlist,
    ot_weightlist, align_weightlist, device,
    dataset=dataset, gpus_per_trial=gpu_per_task, cpus_per_trial=1,
    skip_ot=skip_ot,
)
# save records for visualization purpose
read_csv_path = os.path.join(
        visualize_path,
        f"{source_domain}-{target_domain}-rho{rho_tag}-{prompt_mode}",
    )